# 7 · Host the HalluScan backend on Colab (GPU) + ngrok

Runs the **same** `backend/app.py` as the local demo on a Colab **T4**, exposed at a **fixed ngrok URL**. The Vercel frontend already points at that URL, so there is **nothing to paste** — just keep this notebook running.

Serves **two selectable models** (a model dropdown in the UI): **Llama-3.2-1B** (default — loads first, fast) and **Llama-3.1-8B**. To stay T4-safe only **one model is held in VRAM at a time**; the other is **downloaded to the cache in the background** (`HALLKING_PREFETCH=1`, set in the launch cell) so switching is a quick load, never a mid-demo download. (Set `HALLKING_EAGER=1` instead to keep both in VRAM for instant switching, only if the GPU has room.) Each model runs the **Option-B per-sentence detector** (SEP + HalluShift + TSV + fusion) — a long answer is split into sentences and each factual claim is scored.

**Every session:** fill the two tokens in the CONFIG cell → **Runtime ▸ Restart session and run all**. (The setup cell auto-pulls the latest code; use *Restart session and run all*, not a plain *Run all*, so Python re-imports the updated modules instead of serving cached stale code.) **Keep this tab open** — closing it drops the tunnel.

### 0 · CONFIG — the only cell you edit
Paste your two tokens, then run all. ⚠️ **These tokens are runtime-only — never commit this notebook with them filled in** (the GitHub copy must stay blank). `NGROK_DOMAIN` is the reserved static domain so the URL never changes; leave it as-is.

In [ ]:
# ===== CONFIG (edit the two tokens, then Runtime > Run all) ==============================
REPO            = 'https://github.com/dinitha2004/HalluScan.git'
HF_TOKEN        = ''   # huggingface.co/settings/tokens (Read). Accept BOTH Llama-3.1 + Llama-3.2 licenses.
NGROK_AUTHTOKEN = ''   # dashboard.ngrok.com -> Your Authtoken
NGROK_DOMAIN    = 'declared-angular-matchbox.ngrok-free.dev'  # reserved static domain (leave as-is)
# =======================================================================================
assert HF_TOKEN.strip(), 'Paste HF_TOKEN above (huggingface.co/settings/tokens).'
assert NGROK_AUTHTOKEN.strip(), 'Paste NGROK_AUTHTOKEN above (dashboard.ngrok.com -> Your Authtoken).'
print('config set; domain =', NGROK_DOMAIN or '(random)')

### 1 · Get the code + install deps
The repo is **public**, so no token is needed to clone. (`git clone` makes a `HalluScan/` folder; if it already exists from an earlier run, this cell resets it to the latest `origin/main` so you never serve stale code.) The ML stack is installed explicitly (Colab already has a compatible torch); the backend extras come from `backend/requirements_colab.txt`.

In [ ]:
import os
if not os.path.isdir('HalluScan'):
    !git clone $REPO
else:
    # already cloned this runtime — pull latest so a re-run can't serve stale code
    !git -C HalluScan fetch origin --quiet && git -C HalluScan reset --hard origin/main
%cd HalluScan
!git log -1 --oneline                       # confirm the exact commit being served
# Colab ships a CUDA-matched torch — install the rest explicitly (do NOT reinstall torch).
!pip install -q transformers accelerate bitsandbytes sentencepiece "scikit-learn>=1.6" pandas pyarrow
!pip install -q -r backend/requirements_colab.txt

### 2 · Hugging Face login (gated Llama-3.1 + Llama-3.2)
Uses `HF_TOKEN` from the CONFIG cell. You must have accepted **both** the **Llama-3.1** (for the 8B) and **Llama-3.2** (for the 1B) licenses on Hugging Face. On a fresh runtime the **1B loads first** (~2.5 GB, so the demo is usable in a minute or two); the **8B** (~16 GB) is then downloaded in the **background**, so the first switch to it is a quick load rather than a long download.

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN) if HF_TOKEN.strip() else login()

### 3 · Start the API (background thread) + open the fixed ngrok tunnel
Sets the ngrok authtoken, then opens the tunnel on your **reserved static domain** so the URL is always the same. `nest_asyncio` lets uvicorn run inside Colab's event loop; the model loads on startup (~1–2 min on T4 after weights are cached).

In [ ]:
import os, sys, threading, uvicorn, nest_asyncio
nest_asyncio.apply()
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN.strip()
# Two models, T4-safe: load the 1B first (fast) and DOWNLOAD the 8B to cache in the background, so the dropdown
# switch is a quick load — never a mid-demo download. (Use HALLKING_EAGER='1' instead to keep BOTH in VRAM for
# instant switching, only if the GPU has room.) Must be set BEFORE `import app` (app reads env at import time).
os.environ['HALLKING_PREFETCH'] = '1'
sys.path.insert(0, 'backend'); sys.path.insert(0, 'hallking')
import app  # backend/app.py (FastAPI instance = app.app); lazy default = 1B, 8B prefetched in the background
def _serve():
    uvicorn.run(app.app, host='0.0.0.0', port=8000, log_level='info')
threading.Thread(target=_serve, daemon=True).start()
ngrok.kill()  # drop any tunnel from a previous run before reclaiming the static domain
public = (ngrok.connect(8000, 'http', domain=NGROK_DOMAIN) if NGROK_DOMAIN.strip()
          else ngrok.connect(8000, 'http'))
print('\n========================================================')
print(' PUBLIC URL (the frontend already targets this):')
print('  ', public.public_url)
print('========================================================\n')
# (no ngrok? fallback: !pip install cloudflared && use a cloudflared quick tunnel on port 8000)

### 4 · Wait for model load + smoke-test
Waits for the API to come up — it answers once the **default 1B** has loaded (fast on a fresh runtime; the 8B keeps downloading in the background). It only runs the test questions once `/status` reports `model_loaded`; if it's still loading after the wait, it prints a clear message to re-run this cell — **no error.** Check `regime=sentence`, `current_model=1b`, and the calibrated `t_med`/`t_high` in the status. After it succeeds, open the Vercel site (no pasting needed); switching to the 8B in the dropdown is ready once its background download finishes (watch for the `[HallKing] prefetched '8b'` log).

In [ ]:
import requests, time
deadline = time.time() + 12 * 60   # the API is silent until the default 1B has loaded (fast on a fresh runtime)
ready = None
while time.time() < deadline:
    try:
        s = requests.get('http://localhost:8000/status', timeout=5).json()
        if s.get('model_loaded'):
            ready = s; print('status:', s); break
    except Exception:
        pass   # server not accepting connections yet — still loading the model
    time.sleep(5)

if not ready:
    print('\n[still loading] The default model has not finished loading yet.')
    print('Watch the cell above for the "[HallKing] ready." log, then RE-RUN this cell.')
else:
    for q in ['Who painted the Mona Lisa?', 'Tell me about the Sigiriya rock fortress.']:
        try:
            r = requests.post('http://localhost:8000/infer', json={'question': q}, timeout=180)
            if r.status_code != 200:                      # show the real server error, not a JSON-decode msg
                print(f"\nQ: {q}\n  [infer HTTP {r.status_code}] {r.text[:600]}"); continue
            out = r.json(); agg = out['aggregate']
            print(f"\nQ: {q}\nA: {out['answer'][:160]}")
            print(f"  [{out.get('model','?')}] -> {agg['label']} (fused={agg['fused']}) | {agg['n_flagged']}/{agg['n_sentences']} flagged")
        except Exception as e:
            print(f"\nQ: {q}\n  [infer error] {e}")
    print('\nBackend is live — open the Vercel site (it auto-connects to the fixed URL).')